# Fine-tuning do LLaMA com quantização e LoRA

O modelo fundacional é adaptado ao padrão de respostas institucionais. A quantização em 4 bits reduz o uso de memória; PEFT e LoRA mantêm os pesos originais congelados e treinam pequenas matrizes adicionais. A execução completa requer GPU e acesso autorizado ao modelo base.

In [ ]:
from pathlib import Path
import os, json

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
MODEL_ID = "meta-llama/Llama-2-7b-hf"
OUTPUT_DIR = ROOT / "models/clinical-lora"
RUN_TRAINING = os.getenv("RUN_TRAINING", "0") == "1"
print("Treinamento habilitado:", RUN_TRAINING)

## Dependências e dados

Em ambiente com GPU, instale `pip install -e '.[training]'` na raiz. O dataset já contém o campo `text` no formato instrucional.

In [ ]:
training_available = True
try:
    import torch
    from datasets import load_dataset
    from peft import LoraConfig, prepare_model_for_kbit_training
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from trl import SFTConfig, SFTTrainer
except ImportError as exc:
    training_available = False
    print("Dependências de treinamento não instaladas:", exc)

if training_available:
    dataset = load_dataset("json", data_files={
        "train": str(ROOT / "data/processed/train.jsonl"),
        "test": str(ROOT / "data/processed/test.jsonl"),
    })
    print(dataset)

## Tokenização e quantização

O tokenizador converte texto em tokens. A configuração NF4 representa os pesos em 4 bits e usa ponto flutuante de 16 bits nos cálculos. O `device_map` distribui o modelo pelos dispositivos disponíveis.

In [ ]:
if training_available and RUN_TRAINING:
    quantization_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=os.getenv("HF_TOKEN") or None)
    tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token
    tokenizer.padding_side = "right"
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, quantization_config=quantization_config, device_map="auto",
        token=os.getenv("HF_TOKEN") or None,
    )
    model.config.use_cache = False
    model = prepare_model_for_kbit_training(model)

## Adaptadores e hiperparâmetros

`r` controla o posto das matrizes LoRA; `alpha` escala sua contribuição; `dropout` regulariza o ajuste. A taxa de aprendizado e o número de épocas devem ser avaliados pelas curvas de treino e validação.

In [ ]:
if training_available:
    lora_config = LoraConfig(
        task_type="CAUSAL_LM", r=16, lora_alpha=32, lora_dropout=0.05,
        bias="none", target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    )
    training_config = SFTConfig(
        output_dir=str(OUTPUT_DIR), num_train_epochs=3,
        per_device_train_batch_size=1, per_device_eval_batch_size=1,
        gradient_accumulation_steps=4, learning_rate=2e-4,
        logging_steps=1, eval_strategy="epoch", save_strategy="epoch",
        max_length=512, dataset_text_field="text", report_to="none", seed=42,
    )
    print(lora_config)
    print(training_config)

## Treinamento e salvamento

A célula só executa quando `RUN_TRAINING=1`. O artefato salvo contém os adaptadores e o tokenizador, não uma cópia completa do modelo base.

In [ ]:
if training_available and RUN_TRAINING:
    trainer = SFTTrainer(
        model=model, args=training_config,
        train_dataset=dataset["train"], eval_dataset=dataset["test"],
        processing_class=tokenizer, peft_config=lora_config,
    )
    trainer.train()
    metrics = trainer.evaluate()
    trainer.save_model(str(OUTPUT_DIR))
    tokenizer.save_pretrained(str(OUTPUT_DIR))
    print(metrics)
else:
    print("Configuração validada. Defina RUN_TRAINING=1 em um ambiente com GPU para iniciar o ajuste.")

## Avaliação

Além da perda, compare o modelo base e o adaptado nas mesmas perguntas. Registre respostas incorretas, recusas, fontes ausentes e indícios de memorização. A promoção do adaptador depende dos testes de segurança e de revisão clínica independente.